# Experiment 1: Base Performance (AL)

AL base bug-fix performance is reported as a **supplement sourced from the official BC-Bench leaderboard** (`docs/_data/bug-fix.json`, GitHub Copilot, plain Setting A, 5 runs), not recomputed here. It is the AL side of H1; the Python control is analysed separately. Each row is the leaderboard's mean resolution with 95% CI and pass^5.

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px

# Locate the leaderboard data (repo docs/_data/bug-fix.json)
_root = Path.cwd()
while _root != _root.parent and not (_root / "docs" / "_data" / "bug-fix.json").exists():
    _root = _root.parent
_lb = json.loads((_root / "docs" / "_data" / "bug-fix.json").read_text(encoding="utf-8"))

MODELS = {
    "claude-opus-4-6": "Claude Opus 4.6",
    "claude-sonnet-4-6": "Claude Sonnet 4.6",
    "gpt-5-4": "GPT-5-4",
    "gpt-5-3-codex": "GPT-5-3 Codex",
}

rows = []
for a in _lb["aggregate"]:
    # plain Setting A only (GitHub Copilot, no MCP/experiment)
    if a.get("agent_name") != "GitHub Copilot" or a.get("model") not in MODELS or a.get("experiment"):
        continue
    rows.append({
        "Model": MODELS[a["model"]],
        "Runs": a["num_runs"],
        "Mean Resolved (%)": round(a["average"] * 100, 1),
        "CI low (%)": round(a["ci_low"] * 100, 1),
        "CI high (%)": round(a["ci_high"] * 100, 1),
        "pass^5 (%)": round(a["pass_hat_5"] * 100, 1),
        "Ver": a["benchmark_version"],
    })

order = list(MODELS.values())
supplement_df = pd.DataFrame(rows).sort_values("Model", key=lambda s: s.map({m: i for i, m in enumerate(order)})).reset_index(drop=True)
print("Source: BC-Bench leaderboard (docs/_data/bug-fix.json), GitHub Copilot, plain Setting A, 5 runs")
supplement_df

Source: BC-Bench leaderboard (docs/_data/bug-fix.json), GitHub Copilot, plain Setting A, 5 runs


,Model,Runs,Mean Resolved (%),CI low (%),CI high (%),pass^5 (%),Ver
0,Claude Opus 4.6,5,66.9,64.6,69.7,45.5,0.5.0
1,Claude Sonnet 4.6,5,67.3,66.1,68.5,48.5,0.4.0
2,GPT-5-4,5,58.4,55.8,60.8,37.6,0.3.1
3,GPT-5-3 Codex,5,55.8,54.1,56.8,37.6,0.2.1


In [2]:
# Mean base resolution with 95% CI (from the leaderboard)
fig = px.bar(
    supplement_df, x="Model", y="Mean Resolved (%)", text="Mean Resolved (%)",
    title="AL Base Resolution by Model (BC-Bench leaderboard, 95% CI)",
)
fig.update_traces(
    error_y=dict(
        type="data", symmetric=False,
        array=(supplement_df["CI high (%)"] - supplement_df["Mean Resolved (%)"]).tolist(),
        arrayminus=(supplement_df["Mean Resolved (%)"] - supplement_df["CI low (%)"]).tolist(),
    )
)
fig.update_layout(yaxis_range=[0, 100])
fig.show()